# The Great New York Squirrel Census

In this notebook, we will do an unheard-of thing at Nova IMS. We are going to analye the 2018 Central Park Squirrel Census data.

Make sure you have the ```squirrel_census.csv``` file in your current working environment on lightning.

---

## Setup: Initialize Spark and Load Data

We'll start by setting up the Spark session and loading the data.

In [1]:
!pip install sparksql-magic plotly pandas "nbformat>=4.2.0"

In [2]:
#Checking the installed Java version
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [3]:
!pip install pyspark 

In [4]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:6 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:7 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease          
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease        
Hit:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-security InRelease        
Hit:11 https://security.ubuntu.com/ubuntu noble-security InRelease             
Get:12 http://deb.wakemeops.com/wakemeops stable InRelease [50.5 kB]
Get:13 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble

In [5]:
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [6]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("Where the squirrels at?") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/15 18:19:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/15 18:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
# Define the file path
file_path = "squirrel_census.csv"

# Load the data
df = spark.read.csv(file_path, header=True, inferSchema=True)

<div style="page-break-after: always;"></div>

## Exercise 1: Filtering and Selection (``filter()``, ``select()``, and ``col()``)

The **`filter()`** (or **`where()`**) method subsets rows based on a condition (like SQL's ``WHERE`` clause). The **`select()`** method chooses a subset of columns. To correctly reference columns within these functions, you must use the **`col()`** function, which ensures the column name is interpreted by Spark.

**Task:** Use a combination of **`filter()`** and **`where()`** to find the total number of squirrels observed in the **"AM"** shift AND had **"Gray"** as their ``Primary Fur Color``.

In [15]:
from pyspark.sql.functions import col

# Your solution here

# 1. Filter the DataFrame
filtered_df_ex1 = df.filter(col("Shift") == "AM").where(col("Primary Fur Color") == "Gray")

# 2. Print the total count
print("Total number of AM, Gray squirrels:", filtered_df_ex1.count())

Total number of AM, Gray squirrels: 1087


<div style="page-break-after: always;"></div>

## Exercise 2: Simple Aggregation (``groupBy()``, ``count()``, and ``orderBy()``)

The **`groupBy()`** method partitions data into groups. After grouping, aggregation functions like **`count()`** summarize the data. The **`orderBy()`** method arranges the resulting rows, typically using **`desc()`** for descending order.

**Task:** Use **`groupBy()`** to calculate the number of squirrel sightings for each unique **`Location`** type. Then, use **`orderBy()`** and **`desc()`** to determine the three most common ``Location`` types. Display the ``Location`` and the total ``Count`` in descending order.

In [10]:
from pyspark.sql.functions import desc

# Your solution here

# 1. Group by Location and count
location_counts_df = df.groupBy("Location").count().orderBy(desc("count"))

# 2. Order and show the top 3
location_counts_df.show(3)

+------------+-----+
|    Location|count|
+------------+-----+
|Ground Plane| 2115|
|Above Ground|  843|
|        NULL|   64|
+------------+-----+
only showing top 3 rows


<div style="page-break-after: always;"></div>

## Exercise 3: Calculated Aggregation (``avg()``, ``round()``, and ``agg()``)

The **`agg()`** method computes one or more aggregate functions (like **`avg()`** for the mean) across the data. The **`round()`** function is used to limit decimal places, ensuring clean output for numerical results.

**Task:** Calculate the park-wide **average longitude (X)** and **average latitude (Y)** for squirrels that exhibited the **`Chasing`** behavior (where the ``Chasing`` column is ``true`` or ``TRUE``). Round the resulting averages to 3 decimal places.

In [11]:
from pyspark.sql.functions import avg, round

# Your solution here

# 1. Filter for Chasing squirrels
chasing_squirrels_df = df.filter(col("Chasing") == True)

# 2. Calculate average X and Y, and round the results
avg_coords_df = chasing_squirrels_df.agg(
    round(avg("X"), 3).alias("Avg_Longitude_X"),
    round(avg("Y"), 3).alias("Avg_Latitude_Y")
)

# 3. Show the result
avg_coords_df.show()

+---------------+--------------+
|Avg_Longitude_X|Avg_Latitude_Y|
+---------------+--------------+
|        -73.968|         40.78|
+---------------+--------------+



<div style="page-break-after: always;"></div>

## Exercise 4: Advanced Aggregation - Pivot Table (``pivot()`` and ``dropna()``)

The **`pivot()`** operation rotates the data, turning unique values from one column (e.g., 'Age') into new columns, creating a cross-tabulation. Because pivoting can leave gaps, the **`dropna()`** method is essential for removing null values.

**Task:** Create a pivot table to show the total count of squirrels for each combination of **`Shift`** (``AM``/``PM``), **`Location`** and **`Eating`**. Use **`dropna()`** to ensure NULL values are dropped.

In [12]:
# Your solution here
# (Requires functions already imported in previous exercises)

# 1. Group by Shift and apply the pivot operation on "Eating", aggregating the count
pivot_df = df.groupBy("Shift", "Location").pivot("Eating").count()

# 2. Fill nulls with 0 and show the result
pivot_df = pivot_df.dropna()
pivot_df.show()

+-----+------------+-----+----+
|Shift|    Location|false|true|
+-----+------------+-----+----+
|   AM|Above Ground|  371|  78|
|   PM|Above Ground|  317|  77|
|   AM|Ground Plane|  661| 214|
|   PM|Ground Plane|  861| 379|
+-----+------------+-----+----+



<div style="page-break-after: always;"></div>

## Exercise 5: Aggregation with Booleans
Identify which Hectares have the highest density of vocalizing squirrels (making Kuks, Quaas, or Moans).

1. Group the data by Hectare.
2. Aggregate the data to calculate: (a) Total_Vocal_Sightings: The sum of the Kuks, Quaas, and Moans columns combined. Remember to check the column type and use ```.cast()``` as needed. (b) Total_Sightings: The total number of sightings in that Hectare.
3. Calculate Vocalization_Percentage (Vocal Sightings / Total Sightings * 100).
4. Filter out Hectares with fewer than 10 total sightings.
5. Order the results by the new percentage to find the top 10 loudest areas.

In [13]:
from pyspark.sql.functions import col, count, sum, desc

# Your solution here

# 1. Group by 'Hectare' and calculate the total number of sightings and the sum of each vocalization type.
# By summing the boolean columns ('Kuks', 'Quaas', 'Moans'), we get the count of occurrences for each.
hectare_analysis = df.groupBy("Hectare").agg(
    # Summing the boolean columns directly gives the total count of vocal sightings
    (sum(col("Kuks").cast("int")) + sum(col("Quaas").cast("int")) + sum(col("Moans").cast("int"))).alias("Total_Vocal_Sightings"),

    # Count all sightings in that Hectare
    count("*").alias("Total_Sightings")
)

# 2. Calculate the percentage of sightings that included a funny sound in each Hectare.
# We divide the total vocalizations by the total sightings for the hectare.
final_df = hectare_analysis.withColumn(
    "Vocalization_Percentage",
    (col("Total_Vocal_Sightings") / col("Total_Sightings") * 100)
)

# 3. Filter out any hectares with low sighting counts (less reliable data) and order by percentage.
final_df = final_df.filter(col("Total_Sightings") >= 10)\
                   .orderBy(desc("Vocalization_Percentage"))

# 4. Display the top 10 Hectares with the highest percentage of vocalizing squirrels.
print("Top 10 Hectares by Vocalization Percentage:")
final_df.show(10, truncate=False)

Top 10 Hectares by Vocalization Percentage:
+-------+---------------------+---------------+-----------------------+
|Hectare|Total_Vocal_Sightings|Total_Sightings|Vocalization_Percentage|
+-------+---------------------+---------------+-----------------------+
|33D    |6                    |12             |50.0                   |
|38G    |5                    |11             |45.45454545454545      |
|22C    |7                    |16             |43.75                  |
|38C    |9                    |21             |42.857142857142854     |
|42C    |3                    |10             |30.0                   |
|37H    |3                    |10             |30.0                   |
|04B    |3                    |11             |27.27272727272727      |
|15G    |4                    |18             |22.22222222222222      |
|35C    |3                    |14             |21.428571428571427     |
|36C    |2                    |10             |20.0                   |
+-------+-----------

<div style="page-break-after: always;"></div>

## Exercise 6: DateTime Processing and Visualization (``to_date()``, ``date_format()``, and ``.plot()``)

The **`to_date()`** function converts a date string (like the 'Date' column in this dataset, formatted MMDDYYYY) into a proper Spark `DateType`. The **`date_format()`** function extracts or formats specific date parts. 

**Task:** Analyze the sighting trend over time. Convert the 'Date' column and then use the native plotting API to visualize squirrel sightings per month.

**Steps:**

1.  Use **`to_date()`** to convert the string 'Date' column (format: MMDDYYYY) into a Spark Date type.
2.  Group the data by `ObservationDate` and calculate the total `Count` of sightings.
3.  Plot the `count` column using the native **`.plot.area()`** function.

In [30]:
from pyspark.sql.functions import to_date, col

# Your solution here

# 1. Convert the 'Date' string (MMDDYYYY) to a DateType and extract YearMonth
monthly_sightings_df = df.withColumn(
    "ObservationDate",
    to_date(col("Date").cast("string"), "MMddyyyy")
)

# 2. Aggregate by ``ObservationDate`` 
monthly_counts = monthly_sightings_df.groupBy("ObservationDate").count().orderBy("ObservationDate", ascending=True)

# 5. Plot the area chart using the native pandas-on-Spark plotting API
fig = monthly_counts.plot.area(
    x = "ObservationDate",
    y = "count",
    title="Squirrel Sightings Per Day",
    labels={"count": "Counts", "ObservationDate": "Observation Date"}
)

fig.show()